In [20]:
import os
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, BitsAndBytesConfig

In [21]:
SOURCE = "/home/lisa/Arupreza/AccentFlow-0.2/checkpoints/whisper"
SAVE   = "/home/lisa/Arupreza/AccentFlow-0.2/checkpoints/whisper-nf4"

In [22]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# --- Load + quantize ---
print("[1/3] Loading and quantizing...")
processor = AutoProcessor.from_pretrained(SOURCE)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    SOURCE,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("     Done.")

In [ ]:
# --- Save ---
print(f"[2/3] Saving to {SAVE} ...")
os.makedirs(SAVE, exist_ok=True)
model.save_pretrained(SAVE)
processor.save_pretrained(SAVE)
print("     Saved.")

# --- Verify ---
print("[3/3] Verifying...")
files = os.listdir(SAVE)
size_gb = sum(os.path.getsize(os.path.join(SAVE, f)) for f in files) / 1e9
print(f"     Files: {len(files)} | Size: {size_gb:.2f} GB")
print("     Done.")